# 01_ingest — sources → bronze

Downloads each source, saves the untouched response to the `bronze.raw_docs` volume (a timestamped snapshot), then loads it into a bronze Delta table.
Nothing is cleaned here — bronze is the raw record. Cleaning and leakage removal happen in `02_structured_silver`.

| Part | Source | Bronze table |
|---|---|---|
| 1 | IEG project ratings (labels) | `bronze.ieg_ratings` |
| 2 | World Bank Projects API v3 (project features) | `bronze.projects` |
| 3 | World Bank Indicators API: WDI macro + WGI governance (country context) | `bronze.indicators` |

Documents (PADs etc.) are ingested separately in `01b_ingest_docs`.

**Attach to:** `wb-ml-single`. Run top to bottom.

## Settings

**What:** imports and every configurable value in one place: catalog, raw-file folder, API addresses, page sizes.

**Why:** change a value once here instead of hunting through code; later the job definition can pass these in as parameters.

In [1]:
import requests, json, time, os
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG = "dbw_wb_outcome"
RAW_ROOT = f"/Volumes/{CATALOG}/bronze/raw_docs"
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Part 1 — IEG ratings (the labels)

**IEG** = Independent Evaluation Group, the World Bank's independent evaluation unit. After a project closes, IEG validates the team's self-assessment (ICR Review, ICRR) or does a field evaluation (PPAR, which supersedes the ICRR) and gives a 6-point **outcome** rating. We'll turn Moderately Satisfactory or better into label 1.

Source: Finances One dataset `DS00053` — about 12.6k projects, one row per project (latest evaluation only), refreshed quarterly. The API returns at most 5,000 rows per call, so we page.

⚠️ **Leakage:** several columns are only known after the project ends (`bank_performance`, `quality_at_entry`, `quality_of_supervision`, `me_quality`, `evaluation_*`, `final_closing_fy`, and `country_fcs_*`, which use closing-year status). They stay in bronze and are dropped in silver.

### 1a. Download every page

**What:** requests rows 0–4,999, 5,000–9,999, … until a page comes back short (that's the last one), then checks the total equals the API's own `count`.

**Why:** paging because of the 5,000-row cap; `raise_for_status()` and the `assert` make a failed or partial download stop loudly instead of silently saving half the data; `timeout` stops a hung request; `sleep(1)` is polite to the API.

In [2]:
IEG_URL = "https://datacatalogapi.worldbank.org/dexapps/fone/api/apiservice"
IEG_PARAMS = {"datasetId": "DS00053", "resourceId": "RS00049", "type": "json"}
IEG_PAGE = 5000

def fetch_ieg():
    rows, skip = [], 0
    while True:
        r = requests.get(IEG_URL, params={**IEG_PARAMS, "top": IEG_PAGE, "skip": skip}, timeout=60)
        r.raise_for_status()
        body = r.json()
        batch = body["data"]
        rows.extend(batch)
        print(f"skip={skip}: got {len(batch)}  (total {len(rows)} / {body['count']})")
        if len(batch) < IEG_PAGE:
            break
        skip += IEG_PAGE
        time.sleep(1)
    assert len(rows) == body["count"], "row count mismatch - incomplete download"
    return rows

ieg_rows = fetch_ieg()

skip=0: got 5000  (total 5000 / 12627)
skip=5000: got 5000  (total 10000 / 12627)
skip=10000: got 2627  (total 12627 / 12627)


### 1b. Save the raw response as a timestamped snapshot

**What:** writes all rows to `raw_docs/ieg/ieg_ratings_<UTC timestamp>.json`.

**Why:** this is the untouched original. IEG updates quarterly, so snapshots let you say exactly which version a model was trained on and rebuild without calling the API again.

In [3]:
ieg_dir = f"{RAW_ROOT}/ieg"
os.makedirs(ieg_dir, exist_ok=True)
ieg_raw_path = f"{ieg_dir}/ieg_ratings_{RUN_STAMP}.json"
with open(ieg_raw_path, "w") as f:
    json.dump(ieg_rows, f)
print("saved", ieg_raw_path)

saved /Volumes/dbw_wb_outcome/bronze/raw_docs/ieg/ieg_ratings_20261015T055930Z.json


### 1c. Load into the bronze Delta table

**What:** Spark reads the JSON file, adds `_ingested_at` (load time) and `_source_file`, and saves `bronze.ieg_ratings`.

**Why:** a Delta table is queryable with SQL, versioned, and tracked by Unity Catalog lineage. `multiLine` because the file is one big JSON array. `overwrite` makes re-runs replace rather than duplicate (idempotent); `overwriteSchema` tolerates IEG adding a column.

In [4]:
ieg_df = (spark.read.option("multiLine", True).json(ieg_raw_path)
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(ieg_raw_path)))

(ieg_df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.bronze.ieg_ratings"))

### 1d. Sanity checks

**What:** row count, how many projects per outcome rating, and any project ID that appears more than once.

**Why:** never assume a write worked. These catch missing rows, unexpected label values, and duplicate keys (which would break joins). Expected: ~12.6k rows, Satisfactory largest, ~150 Not Rated, one duplicate ID.

In [5]:
ieg_t = spark.table(f"{CATALOG}.bronze.ieg_ratings")
print("rows:", ieg_t.count())
display(ieg_t.groupBy("outcome").count().orderBy(F.desc("count")))
display(ieg_t.groupBy("project_id").count().filter("count > 1"))

rows: 12627


outcome,count
Satisfactory,5638
Moderately Satisfactory,3153
Unsatisfactory,1870
Moderately Unsatisfactory,1223
Highly Satisfactory,398
Highly Unsatisfactory,194
Not Rated,151


project_id,count
P122764,2


# Part 2 — Projects API (project features)

Source: `search.worldbank.org/api/v3/projects` — about 28k projects, at most 1,000 per call. (v2 stopped updating in Dec 2024, so we use v3.) Every IEG-rated project has a match here.

⚠️ **Leakage:** `closingdate` is the *current* closing date (extensions signal trouble), `status` reflects today, and the effectiveness / ICR milestone dates come after approval. `pdo` (development objective) can be rewritten during restructuring. All kept in bronze, handled in silver.

### 2a. Download every page, saving each page as it arrives

**What:** requests projects 0–999, 1,000–1,999, … up to the API's `total`, writing each page to its own file in a timestamped folder `raw_docs/projects/<UTC timestamp>/`. Then checks the count.

**Why:** `fl` asks only for the fields we may use (~53 MB instead of ~270 MB). The API returns projects as a dict keyed by ID, so we convert to a list of records for Spark. Page-by-page saving keeps memory low and keeps finished pages if a later request fails. Takes ~2–4 minutes.

In [6]:
PROJ_URL = "https://search.worldbank.org/api/v3/projects"
PROJ_PAGE = 1000
PROJ_FIELDS = [
    "id", "project_name", "countryshortname", "countrycode", "regionname",
    "boardapprovaldate", "fiscalyear", "status", "prodline_exact",
    "totalamt", "grantamt", "idacommamt", "lendprojectcost",
    "major_sector_name", "sector_name", "sector_percent", "themecode",
    "borrower", "impagency", "envassesmentcategorycode", "milestones",
    "pdo", "project_abstract", "parentprojid", "closingdate",
]
proj_dir = f"{RAW_ROOT}/projects/{RUN_STAMP}"

def fetch_projects(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    offset, total, n = 0, None, 0
    while total is None or offset < total:
        r = requests.get(PROJ_URL, timeout=120, params={
            "format": "json", "rows": PROJ_PAGE, "os": offset,
            "fl": ",".join(PROJ_FIELDS)})
        r.raise_for_status()
        body = r.json()
        total = int(body["total"])
        page = list(body["projects"].values())
        if not page:
            break
        with open(f"{out_dir}/page_{offset:06d}.json", "w") as f:
            json.dump(page, f)
        n += len(page)
        print(f"os={offset}: {len(page)}  (so far {n} / {total})")
        offset += PROJ_PAGE
        time.sleep(1)
    assert n == total, f"got {n} projects, expected {total}"
    return n

fetch_projects(proj_dir)

os=0: 1000  (so far 1000 / 28184)
os=1000: 1000  (so far 2000 / 28184)
os=2000: 1000  (so far 3000 / 28184)
os=3000: 1000  (so far 4000 / 28184)
os=4000: 1000  (so far 5000 / 28184)
os=5000: 1000  (so far 6000 / 28184)
os=6000: 1000  (so far 7000 / 28184)
os=7000: 1000  (so far 8000 / 28184)
os=8000: 1000  (so far 9000 / 28184)
os=9000: 1000  (so far 10000 / 28184)
os=10000: 1000  (so far 11000 / 28184)
os=11000: 1000  (so far 12000 / 28184)
os=12000: 1000  (so far 13000 / 28184)
os=13000: 1000  (so far 14000 / 28184)
os=14000: 1000  (so far 15000 / 28184)
os=15000: 1000  (so far 16000 / 28184)
os=16000: 1000  (so far 17000 / 28184)
os=17000: 1000  (so far 18000 / 28184)
os=18000: 1000  (so far 19000 / 28184)
os=19000: 1000  (so far 20000 / 28184)
os=20000: 1000  (so far 21000 / 28184)
os=21000: 1000  (so far 22000 / 28184)
os=22000: 1000  (so far 23000 / 28184)
os=23000: 1000  (so far 24000 / 28184)
os=24000: 1000  (so far 25000 / 28184)
os=25000: 1000  (so far 26000 / 28184)
os=26000

28184

### 2b. Load all pages into the bronze Delta table

**What:** Spark reads every page file at once (`*.json`), infers columns, adds `_ingested_at` and `_source_file` (from Spark's hidden `_metadata.file_path`), and saves `bronze.projects`.

**Why:** one table for all projects; `_source_file` lets you trace any odd row back to its raw page. No cleaning yet — amounts stay as text and `milestones` stays nested.

In [7]:
proj_df = (spark.read.option("multiLine", True).json(f"{proj_dir}/*.json")
           .withColumn("_ingested_at", F.current_timestamp())
           .withColumn("_source_file", F.col("_metadata.file_path")))

(proj_df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.bronze.projects"))

### 2c. Sanity checks — does the join key work?

**What:** row and distinct-ID counts, then how many rated IEG projects have a matching project row (a `left_semi` join keeps IEG rows that match, without adding columns), plus 5 sample rows.

**Why:** proves `project_id` ↔ `id` joins cleanly *now*, instead of discovering a format mismatch in gold. Expected: ~28.2k rows, distinct = rows, matched ≈ all rated IEG projects (~12.5k).

In [8]:
proj_t = spark.table(f"{CATALOG}.bronze.projects")
print("rows:", proj_t.count(), "| distinct ids:", proj_t.select("id").distinct().count())

ieg_rated = (spark.table(f"{CATALOG}.bronze.ieg_ratings")
             .filter("outcome != 'Not Rated'").select("project_id").distinct())
matched = ieg_rated.join(proj_t, ieg_rated.project_id == proj_t.id, "left_semi").count()
print(f"IEG-rated projects found in projects table: {matched} / {ieg_rated.count()}")

display(proj_t.select("id", "boardapprovaldate", "totalamt", "major_sector_name", "pdo").limit(5))

rows: 28184 | distinct ids: 28184
IEG-rated projects found in projects table: 12475 / 12475


id,boardapprovaldate,totalamt,major_sector_name,pdo
P518248,2027-10-26T00:00:00Z,85000000,,The project development objective to improve irrigation services and enhance agricultural productivity in the Gihanga III area.
P513206,2027-09-30T00:00:00Z,647000000,"Education,Health,Social Protection",To improve healthy behaviors and simultaneous utilization by the target population of nutrition-specific and nutrition-sensitive interventions known to reduce stunting.
P518265,2027-09-28T00:00:00Z,150000000,,To catalyze investment in circular agribusiness solutions to create jobs and reduce GHG emissions in targeted rural communities in selected provinces and counties of the Yellow River Basin.
P510631,2027-09-21T00:00:00Z,610000000,"Agriculture, Fishing and Forestry,Public Administration,Social Protection,Social Sustainability and Inclusion,Water, Sanitation and Waste Management",To improve flood management and strengthen resilience to water-related climate risks in the Lower Chao Phraya Basin
P518954,2027-09-16T00:00:00Z,50000000,,"To increase the productivity and market integration of selected agricultural value chains in Mauritania, with a focus on smallholder farmers, women, and youth, and to mobilize private investment in agribusiness."


# Part 3 — Country indicators (macro + governance context)

Source: World Bank Indicators API `api.worldbank.org/v2` (the same API the `wbgapi` package wraps — calling it directly means no extra install in scheduled jobs).

| Code | Meaning | Source |
|---|---|---|
| `NY.GDP.PCAP.KD` | GDP per capita, constant 2015 US$ | WDI (2) |
| `NY.GDP.MKTP.KD.ZG` | GDP growth, annual % | WDI (2) |
| `FP.CPI.TOTL.ZG` | Inflation, consumer prices % | WDI (2) |
| `DT.DOD.DECT.GN.ZS` | External debt stocks, % of GNI | WDI (2) |
| `SH.XPD.CHEX.GD.ZS` | Health expenditure, % of GDP | WDI (2) |
| `SE.XPD.TOTL.GD.ZS` | Government education spending, % of GDP | WDI (2) |
| `SP.POP.TOTL` | Population | WDI (2) |
| `NE.TRD.GNFS.ZS` | Trade, % of GDP | WDI (2) |
| `BX.KLT.DINV.WD.GD.ZS` | FDI net inflows, % of GDP | WDI (2) |
| `GOV_WGI_{GE,RQ,RL,CC,PV,VA}.EST` | Worldwide Governance Indicators (≈ −2.5 … +2.5): government effectiveness, regulatory quality, rule of law, control of corruption, political stability, voice & accountability | WGI (3) |

Note: the old WGI codes (`GE.EST`, …) were archived; the current ones are `GOV_WGI_*`. WGI starts in 1996 and was every two years until 2002.

### 3a. Download each indicator for all countries, 1980–2025

**What:** for each code, pages through `country/all/indicator/<code>` (5,000 rows per page) and saves the raw JSON per indicator under `raw_docs/indicators/<UTC timestamp>/`.

**Why:** one file per indicator keeps the raw record traceable; the API rejects very large pages, so 5,000 is a safe size. We fetch from 1980 so that "the year before approval" exists even for early projects.

In [9]:
IND_URL = "https://api.worldbank.org/v2/country/all/indicator/{code}"
WDI_CODES = ["NY.GDP.PCAP.KD", "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "DT.DOD.DECT.GN.ZS",
             "SH.XPD.CHEX.GD.ZS", "SE.XPD.TOTL.GD.ZS", "SP.POP.TOTL", "NE.TRD.GNFS.ZS",
             "BX.KLT.DINV.WD.GD.ZS"]
WGI_CODES = [f"GOV_WGI_{x}.EST" for x in ["GE", "RQ", "RL", "CC", "PV", "VA"]]
IND_SOURCES = {**{c: 2 for c in WDI_CODES}, **{c: 3 for c in WGI_CODES}}
ind_dir = f"{RAW_ROOT}/indicators/{RUN_STAMP}"

def fetch_indicator(code, source):
    rows, page = [], 1
    while True:
        r = requests.get(IND_URL.format(code=code), timeout=120, params={
            "format": "json", "per_page": 5000, "page": page, "date": "1980:2025", "source": source})
        r.raise_for_status()
        meta, data = r.json()
        rows.extend(data or [])
        if page >= int(meta["pages"]):
            break
        page += 1
        time.sleep(0.5)
    assert len(rows) == int(meta["total"]), f"{code}: got {len(rows)}, expected {meta['total']}"
    return rows

os.makedirs(ind_dir, exist_ok=True)
for code, source in IND_SOURCES.items():
    rows = fetch_indicator(code, source)
    with open(f"{ind_dir}/{code}.json", "w") as f:
        json.dump(rows, f)
    print(f"{code}: {len(rows)} rows")

NY.GDP.PCAP.KD: 12236 rows
NY.GDP.MKTP.KD.ZG: 12236 rows
FP.CPI.TOTL.ZG: 12236 rows
DT.DOD.DECT.GN.ZS: 12236 rows
SH.XPD.CHEX.GD.ZS: 12236 rows
SE.XPD.TOTL.GD.ZS: 12236 rows
SP.POP.TOTL: 12236 rows
NE.TRD.GNFS.ZS: 12236 rows
BX.KLT.DINV.WD.GD.ZS: 12236 rows
GOV_WGI_GE.EST: 5832 rows
GOV_WGI_RQ.EST: 5832 rows
GOV_WGI_RL.EST: 5832 rows
GOV_WGI_CC.EST: 5832 rows
GOV_WGI_PV.EST: 5832 rows
GOV_WGI_VA.EST: 5832 rows


### 3b. Load into one long bronze table

**What:** reads all indicator files and keeps one row per (indicator, country, year, value), flattening the nested `indicator` / `country` objects into plain columns.

**Why:** "long" format (one value per row) is how the API delivers it and is easy to append to; the pivot to one-row-per-country-year happens in silver. Flattening is not cleaning — every value is kept as delivered, nulls included.

In [10]:
ind_df = (spark.read.option("multiLine", True).json(f"{ind_dir}/*.json")
          .select(F.col("indicator.id").alias("indicator_id"),
                  F.col("indicator.value").alias("indicator_name"),
                  F.col("country.id").alias("country_iso2"),
                  F.col("countryiso3code").alias("country_iso3"),
                  F.col("country.value").alias("country_name"),
                  F.col("date").alias("year"),
                  F.col("value").cast("double").alias("value"))
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))

(ind_df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.bronze.indicators"))

### 3c. Sanity checks

**What:** per indicator: rows, non-null values, and first/last year with data.

**Why:** shows coverage at a glance — e.g. WGI should start in 1996 and health spending around 2000 — so we know which features will be missing for older projects.

In [11]:
ind_t = spark.table(f"{CATALOG}.bronze.indicators")
ind_summary = (ind_t.groupBy("indicator_id")
               .agg(F.count("*").alias("rows"), F.count("value").alias("non_null"),
                    F.min(F.when(F.col("value").isNotNull(), F.col("year"))).alias("first_year"),
                    F.max(F.when(F.col("value").isNotNull(), F.col("year"))).alias("last_year"))
               .orderBy("indicator_id"))
display(ind_summary)

indicator_id,rows,non_null,first_year,last_year
BX.KLT.DINV.WD.GD.ZS,12236,10425,1980,2025
DT.DOD.DECT.GN.ZS,12236,5365,1980,2024
FP.CPI.TOTL.ZG,12236,9477,1980,2025
GOV_WGI_CC.EST,5832,5588,1996,2025
GOV_WGI_GE.EST,5832,5553,1996,2025
GOV_WGI_PV.EST,5832,5644,1996,2025
GOV_WGI_RL.EST,5832,5691,1996,2025
GOV_WGI_RQ.EST,5832,5554,1996,2025
GOV_WGI_VA.EST,5832,5652,1996,2025
NE.TRD.GNFS.ZS,12236,9071,1980,2025


## Run summary

**What:** returns the key counts as the notebook's exit value.

**Why:** when this notebook runs as a job task, the exit value shows up in the run output, so a scheduled run can be checked without opening the notebook.

In [12]:
summary = {
    "ieg_rows": ieg_t.count(),
    "projects_rows": proj_t.count(),
    "ieg_rated_matched": matched,
    "indicator_rows": ind_t.count(),
    "indicators": ind_t.select("indicator_id").distinct().count(),
}
print(summary)
dbutils.notebook.exit(json.dumps(summary))

{"ieg_rows": 12627, "projects_rows": 28184, "ieg_rated_matched": 12475, "indicator_rows": 145116, "indicators": 15}